In [1]:
import json
import huggingface_hub

with open(r'D:\PythonWorkspace\operator_kayoko\config.json', 'r') as js:
    HF_TOKEN = json.load(js)
    HF_TOKEN = HF_TOKEN['HUGGINGFACE_TOKEN']

huggingface_hub.login(HF_TOKEN)

d:\anaconda3\envs\koalpaca\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Dataset 1
'''
ko-en파일을 내려받은 KDE4의 데이터셋 사이트.
데이터셋 확인 결과 아래 절차가 필요.
    1) 순수 ko | en 언어로 정제가 필요
    2) 특수문자가 있는 행데이터 삭제
    3) "& kde;" 내용 삭제

https://opus.nlpl.eu/legacy/KDE4.php
https://velog.io/@kksj0216/Machine-Translation-with-Hugging-Face
'''

# Dataset 2
'''
Huggingface에서 bongsoo/news_talk_en_ko 데이터셋을 활용.

https://metamath1.github.io/blog/posts/gentle-t5-trans/gentle_t5_trans.html?utm_source=pytorchkr&ref=pytorchkr
'''

'\nHuggingface에서 bongsoo/news_talk_en_ko 데이터셋을 활용.\n\nhttps://metamath1.github.io/blog/posts/gentle-t5-trans/gentle_t5_trans.html?utm_source=pytorchkr&ref=pytorchkr\n'

# 1. Dataset preview

In [5]:
from datasets import load_dataset

raw_dataset = load_dataset('kde4', lang1='en', lang2='ko')
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 76708
    })
})

In [6]:
splited_dataset = raw_dataset['train'].train_test_split(train_size=0.7, seed=42)
splited_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 53695
    })
    test: Dataset({
        features: ['id', 'translation'],
        num_rows: 23013
    })
})

In [14]:
splited_dataset['train'][13]

{'id': '30398', 'translation': {'en': 'Coding', 'ko': '코딩'}}

In [15]:
# Tokenizing
from transformers import AutoTokenizer

model_ckpt = 'sohyun416/marian-finetuned-kde4-ko-to-en'
tokenizer = AutoTokenizer.from_pretrained(model_ckpt, return_tensors='pt')

d:\anaconda3\envs\koalpaca\lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\J\.cache\huggingface\hub\models--sohyun416--marian-finetuned-kde4-ko-to-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\anaconda3\envs\koalpaca\lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserW

In [19]:
idnum = 20
en_sen = splited_dataset['train'][idnum]['translation']['en']
ko_sen = splited_dataset['train'][idnum]['translation']['ko']

inputs = tokenizer(ko_sen, text_target=en_sen)
print(inputs)
print(tokenizer.convert_ids_to_tokens(inputs['input_ids']))
print(tokenizer.convert_ids_to_tokens(inputs['labels']))



{'input_ids': [9, 12927, 5639, 10730, 53, 4902, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1], 'labels': [57, 10579, 6612, 13, 49449, 0]}
['▁', '픽', '셀', '▁단위', '의', '▁높이', '</s>']
['▁The', '▁desired', '▁height', '▁in', '▁pixels', '</s>']


In [21]:
# Prep func.
def preprocessing_sentences(sentences):
    arrival = [sen['ko'] for sen in sentences['translation']]
    destination = [sen['ko'] for sen in sentences['translation']]
    model_inputs = tokenizer(arrival, text_target=destination, max_length=128, truncation=True)
    return model_inputs
    
tokenized_dataset = splited_dataset.map(
    preprocessing_sentences,
    batched=True,
    remove_columns=splited_dataset['train'].column_names,
)

tokenized_dataset

Map: 100%|██████████| 23013/23013 [00:04<00:00, 5047.79 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 53695
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 23013
    })
})

# 2. Fine-tune

In [22]:
from transformers import AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained("sohyun416/marian-finetuned-kde4-ko-to-en")  

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
batch = data_collator([tokenized_dataset['train'][i] for i in range(1, 4)])
print(batch.keys())
print(batch['labels'])  # -100 means padding truncation over maximum length

dict_keys(['input_ids', 'attention_mask', 'labels', 'decoder_input_ids'])
tensor([[    9, 34377,     0,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100],
        [    9,   351,   500,     9,     1,     9,    97,  1765,     9,   270,
             9,  4810,   805,     0],
        [ 6358,  3202,     0,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100]])


In [ ]:
# https://velog.io/@kksj0216/Machine-Translation-with-Hugging-Face